# Upload orders to Snowflake

Select the project's **venv** Python kernel, then run the cells in order or choose **Run All**.

This notebook uploads `data/clean/orders.csv` and recreates `RETAIL_DB.INCOMING.ORDERS_PYTHON_TEST` using the structure of `RETAIL_DB.CORE.ORDERS`. Each load replaces the test table's contents.


In [ ]:
import os
import sys
from pathlib import Path

import snowflake.connector
from cryptography.hazmat.primitives import serialization
from dotenv import load_dotenv

# Notebooks have no __file__. Locate the project from the working directory.
REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "upload_orders.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the shopify-retail-analytics project.")

ORDERS_FILE = REPO_ROOT / "data" / "clean" / "orders.csv"
STAGE_PATH = "@~/retail_python_dev/"
TEST_TABLE = "RETAIL_DB.INCOMING.ORDERS_PYTHON_TEST"

if not ORDERS_FILE.is_file():
    raise FileNotFoundError(f"Orders CSV not found: {ORDERS_FILE}")
print("Python:", sys.executable)
print("Orders file:", ORDERS_FILE)


## Load connection settings
Credentials are read from `.env`; they are not displayed or stored in notebook outputs.


In [ ]:
def connection_settings():
    """Load configuration and decode the private key for authentication."""
    load_dotenv(REPO_ROOT / ".env")

    required = (
        "SNOWFLAKE_ACCOUNT",
        "SNOWFLAKE_USER",
        "SNOWFLAKE_PRIVATE_KEY_PATH",
    )
    missing = [name for name in required if not os.environ.get(name)]

    if missing:
        raise ValueError(
            f"Missing settings in .env or environment: {', '.join(missing)}"
        )

    key_path = Path(os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"]).expanduser()
    if not key_path.is_absolute():
        key_path = REPO_ROOT / key_path

    passphrase = os.environ.get("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE")
    private_key = serialization.load_pem_private_key(
        key_path.read_bytes(),
        password=passphrase.encode() if passphrase else None,
    )

    settings = {
        "account": os.environ["SNOWFLAKE_ACCOUNT"],
        "user": os.environ["SNOWFLAKE_USER"],
        "authenticator": "SNOWFLAKE_JWT",
        "private_key": private_key.private_bytes(
            serialization.Encoding.DER,
            serialization.PrivateFormat.PKCS8,
            serialization.NoEncryption(),
        ),
    }

    for setting in ("role", "warehouse", "database", "schema"):
        value = os.environ.get(f"SNOWFLAKE_{setting.upper()}")
        if value:
            settings[setting] = value

    settings.setdefault("warehouse", "COMPUTE_WH")
    return settings


settings = connection_settings()
print("Configuration and private key loaded.")


## Upload the CSV
Replace the staged copy with the current local file and list the stage.


In [ ]:
with snowflake.connector.connect(**settings) as conn:
    with conn.cursor() as cursor:
        cursor.execute("SELECT CURRENT_USER(), CURRENT_ROLE()")
        print("Connected as:", cursor.fetchone())
        local_path = ORDERS_FILE.as_posix().replace("'", "''")
        cursor.execute(
            f"PUT 'file://{local_path}' {STAGE_PATH} "
            "AUTO_COMPRESS = FALSE OVERWRITE = TRUE"
        )
        print("Upload:", cursor.fetchall())
        cursor.execute(f"LIST {STAGE_PATH}")
        print("Staged files:", cursor.fetchall())


## Load and verify the test table
Recreate the test table, load `orders.csv` with the existing CSV format, and count the rows.


In [ ]:
with snowflake.connector.connect(**settings) as conn:
    with conn.cursor() as cursor:
        cursor.execute(
            f"CREATE OR REPLACE TABLE {TEST_TABLE} LIKE RETAIL_DB.CORE.ORDERS"
        )
        cursor.execute(f"""
            COPY INTO {TEST_TABLE}
            FROM {STAGE_PATH}
            FILES = ('orders.csv')
            FILE_FORMAT = (FORMAT_NAME = 'RETAIL_DB.CORE.CLEAN_CSV_FORMAT')
            ON_ERROR = 'ABORT_STATEMENT'
        """)
        print("Table load:", cursor.fetchall())
        cursor.execute(f"SELECT COUNT(*) FROM {TEST_TABLE}")
        print("Rows in ORDERS_PYTHON_TEST:", cursor.fetchone()[0])
print("Orders upload and test load completed.")
